# Phase 1 — Notebook 1: Preprocessing

SQL extract → clean, filtered token corpus. Pure wrangling — no analytical work.

| Step | What | Key output |
|------|------|------------|
| 1 | Ingest | `prepared/01_ingested.parquet` |
| 2 | Token normalization | `prepared/02_lemmatized.parquet` |
| 3+4 | Consolidation (auto-build + apply) | `prepared/04_consolidated.parquet`, `CONFIG/consolidation_map.csv` |
| 5 | Sparsity filter | `prepared/05_filtered.parquet`, `vocab/unigram_*.csv` |
| 6 | Quality CP1 | `quality/quality_cp1.json` |


## Setup

In [1]:
import sys, re
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import load_cfg, ingest, flat_freq, token_doc_freq, quality_report

CFG = load_cfg()

def OUT(subdir, fname):
    """Resolve an OUTPUTS path and create it."""
    p = ROOT / "OUTPUTS" / subdir / fname
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

print("Config:", CFG)

Config: {'sql': {'min_doc_count': 25, 'max_doc_fraction': 0.5, 'token_cap': 40}, 'preprocess': {'min_project_tokens': 5, 'dedup_tokens_per_project': True}, 'ngrams': {'max_n': 3, 'min_freq': 10}, 'tfidf': {'min_df': 10, 'max_df': 0.8, 'ngram_range': [1, 2]}, 'nmf': {'top_words': 12, 'random_seed': 42, 'max_iter': 400}, 'analysis': {'group_by': ['project_category'], 'min_category_projects': 50, 'min_coverage': 6, 'bins': [], 'group_descriptions': {'project_category': 'product categories teachers use to request classroom supplies on DonorsChoose', 'grade_band': 'grade bands (e.g. PreK-2, 3-5, 6-8, 9-12) across all DonorsChoose project requests', 'posted_year': 'annual time periods showing how teacher request language has evolved over time on DonorsChoose', 'posted_year_quarter': 'quarterly time periods showing how teacher request language has evolved over time on DonorsChoose', 'metro_type_at_time_of_posting': 'school metro types (urban, suburban, rural) across DonorsChoose project reque

---
## Step 1 — Ingest

`tokens` arrives as a comma-separated LISTAGG string from SQL; `ingest()`
splits it into a list. `theme_version` is planted here for Phase 2+ registry
continuity.


In [2]:
DATA_DIR   = ROOT / "DATA"
CHUNK_SIZE = 500_000   # rows per chunk; tune down if memory is tight

# ── 1. Union all project_essay* CSVs via chunked reads ───────────────────────
essay_files = sorted(DATA_DIR.glob("project_essay*.csv"))
if not essay_files:
    raise FileNotFoundError(f"No project_essay*.csv files found in {DATA_DIR}")
print(f"Found {len(essay_files)} essay file(s): {[f.name for f in essay_files]}")

chunks = []
for fpath in essay_files:
    date_cols = [c for c in pd.read_csv(fpath, nrows=0).columns
                 if c in {"posted_date", "funded_date"}]
    for chunk in pd.read_csv(fpath, chunksize=CHUNK_SIZE,
                             parse_dates=date_cols or False):
        chunk["tokens"] = (chunk["tokens"].fillna("").str.split(",")
                           .apply(lambda ts: [t.strip() for t in ts if t.strip()]))
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"  Raw union: {len(df):,} rows")

# ── 2. Left-join project attributes ──────────────────────────────────────────
ATTR_CSV = DATA_DIR / "project_attributes.csv"
if not ATTR_CSV.exists():
    raise FileNotFoundError(f"project_attributes.csv not found in {DATA_DIR}")

attrs = pd.read_csv(ATTR_CSV)
df    = df.merge(attrs, on="project_id", how="left")
df["posted_date"] = pd.to_datetime(df["posted_date"])
df["posted_year"] = df["posted_date"].dt.year
df["posted_year_quarter"] = (df["posted_date"].dt.year.astype(str) + "_Q" + df["posted_date"].dt.quarter.astype(str))
print(f"  After attribute join: {len(df):,} rows  |  columns: {list(df.columns)}")

df["theme_version"] = "1.0"
df.to_parquet(OUT("prepared", "01_ingested.parquet"), index=False)
print(f"\nSaved 01_ingested.parquet")

Found 10 essay file(s): ['project_essays_01.csv', 'project_essays_02.csv', 'project_essays_03.csv', 'project_essays_04.csv', 'project_essays_05.csv', 'project_essays_06.csv', 'project_essays_07.csv', 'project_essays_08.csv', 'project_essays_09.csv', 'project_essays_10.csv']
  Raw union: 894,117 rows
  After attribute join: 896,295 rows  |  columns: ['project_id', 'tokens', 'funded_date', 'materials_vendors_count', 'material_cost', 'has_match_on_posting', 'fy25_historical_efs_status', 'expiration_date', 'is_favorite_or_exciting', 'is_professional_development', 'is_student_led', 'is_teachers_first_posted_project', 'teachers_nth_posted_project', 'project_received_government_grant', 'percentage_free_lunch_at_time_of_posting', 'metro_type_at_time_of_posting', 'school_zip', 'school_id_at_time_of_posting', 'school_enrollment', 'school_is_historically_underrepresented_race', 'school_is_low_income', 'school_is_racially_predominant', 'school_is_underserved_rural', 'school_year_open', 'school_per

---
## Step 2 — Token Normalization

Uses `simplemma` for morphological normalization — handles inflected forms
(`drives → drive`, `organized → organize`, `lives → live`) correctly.
Domain terms and proper nouns it does not recognise are returned unchanged;
the consolidation map handles any residual cases.
Also update the table in cell 0: output filename stays `02_lemmatized.parquet`.

In [3]:
import simplemma

def normalize_tokens(tokens):
    """Light morphological normalization via simplemma.
    Handles inflected forms (drives→drive, organized→organize, lives→live)
    correctly without bad stems from hand-rolled suffix rules.
    Domain terms and proper nouns simplemma does not recognise are returned
    unchanged; the consolidation map handles any residual cases.
    """
    return [simplemma.lemmatize(t, lang="en") for t in tokens]

df_raw       = df.copy()
df["tokens"] = df["tokens"].apply(normalize_tokens)
df.to_parquet(OUT("prepared", "02_lemmatized.parquet"), index=False)

before  = flat_freq(df_raw)
after   = flat_freq(df)
changed = (after.rename("after").to_frame()
             .join(before.rename("before"), how="outer")
             .fillna(0).astype(int)
             .query("after != before")
             .assign(delta=lambda x: x.after - x.before))
changed.sort_values("delta").head(15)

,after,before,delta
students,0,5038587,-5038587
learning,354,1276697,-1276343
reading,2423,574509,-572086
skills,0,499047,-499047
books,0,490796,-490796
materials,0,411231,-411231
children,0,267094,-267094
activities,0,252611,-252611
needs,0,247203,-247203
supplies,0,244873,-244873


---
## Steps 3 + 4 — Consolidation: Auto-build & Apply

Builds a consolidation map fully automatically — no human review step.

**Two automatic passes:**
1. **Known-fix dictionary** — corrects documented simplemma truncation artifacts
   (e.g. `creat → create`, `organiz → organize`). Extend `KNOWN_FIXES` to add
   any domain-specific corrections over time.
2. **Plural-pair detection** — within the top-N vocab, maps `word + s/es → word`
   only when the singular form also appears in the same vocab window.

Writes `CONFIG/consolidation_map.csv` (for audit trail) and
`OUTPUTS/prepared/consolidation_candidates.csv` (all top-N tokens with auto-filled replacements).
Then applies the map to `df` in-place.


In [4]:
_TRUNC_RE = re.compile(r"(at|iv|iz|az|ig|if|ic|olog)$")

# ── Known simplemma truncation fixes ─────────────────────────────────────────
# Extend this dict if you find new bad stems in the quality report.
KNOWN_FIXES = {
    "creat":     "create",
    "organiz":   "organize",
    "writ":      "write",
    "engag":     "engage",
    "inspir":    "inspire",
    "challeng":  "challenge",
    "experienc": "experience",
    "includ":    "include",
    "improv":    "improve",
    "achiev":    "achieve",
    "contribut": "contribute",
    "encourag":  "encourage",
    "provid":    "provide",
    "requir":    "require",
    "recogniz":  "recognize",
    "utiliz":    "utilize",
    "analyz":    "analyze",
    "communic":  "communicate",
    "particip":  "participate",
    "collabor":  "collaborate",
    "introduc":  "introduce",
    "practic":   "practice",
    "explor":    "explore",
    "prepar":    "prepare",
    "develop":   "develop",   # simplemma sometimes returns "develop" correctly, kept as no-op guard
}

def _auto_replacement(original: str, token_set: set) -> str:
    """Return replacement string, or '' to leave token unchanged."""
    if original in KNOWN_FIXES:
        return KNOWN_FIXES[original]
    # Simple plural collapse: only when singular is also in the top-N vocab
    if original.endswith("s") and not original.endswith("ss") and len(original) > 3:
        singular = original[:-1]
        if singular in token_set:
            return singular
    if original.endswith("es") and len(original) > 4:
        singular = original[:-2]
        if singular in token_set:
            return singular
    return ""

def build_and_apply_consolidation(df, top_n=1000):
    vocab = flat_freq(df).head(top_n).reset_index()
    vocab.columns = ["original", "freq"]
    vocab["flag_type"] = vocab["original"].apply(
        lambda w: "truncation" if len(w) <= 8 and _TRUNC_RE.search(w) else ""
    )
    token_set = set(vocab["original"])
    vocab["replacement"] = vocab["original"].apply(
        lambda w: _auto_replacement(w, token_set)
    )
    vocab["notes"] = "auto"
    return vocab

# Build map
candidates = build_and_apply_consolidation(df)
MAP_PATH = ROOT / "CONFIG/consolidation_map.csv"
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(OUT("prepared", "consolidation_candidates.csv"), index=False)
candidates.to_csv(MAP_PATH, index=False)
print(f"{len(candidates)} tokens scanned  |  "
      f"{(candidates.replacement != '').sum()} auto-mapped  |  "
      f"{(candidates.flag_type == 'truncation').sum()} truncation-flagged")

# Apply map
cmap = candidates[candidates["replacement"] != ""].copy()
mapping = dict(zip(cmap["original"].str.strip(), cmap["replacement"].str.strip()))
df["tokens"] = df["tokens"].apply(lambda ts: [mapping.get(t, t) for t in ts])
cmap[["original", "replacement", "flag_type", "notes"]].to_csv(
    OUT("prepared", "consolidation_audit.csv"), index=False
)
df.to_parquet(OUT("prepared", "04_consolidated.parquet"), index=False)

WATCH = {"creat","engag","experienc","organiz","includ","writ","inspir","challeng"}
found = flat_freq(df)[lambda s: s.index.isin(WATCH)]
print(f"Mappings applied: {len(mapping)}")
print("✓ No watched stems remain" if found.empty
      else f"⚠ Still present:\n{found.to_string()}")
candidates[candidates["replacement"] != ""].head(20)


1000 tokens scanned  |  6 auto-mapped  |  17 truncation-flagged
Mappings applied: 6
✓ No watched stems remain


,original,freq,flag_type,replacement,notes
93,develop,138492,,develop,auto
287,ipads,54257,,ipad,auto
316,means,51206,,mean,auto
608,towards,25607,,toward,auto
719,bags,21052,,bag,auto
844,whiteboards,17582,,whiteboard,auto


---
## Step 5 — Sparsity Filter

Applies corpus-level document-frequency bounds from `params.yaml`. Projects
below `min_project_tokens` are written to a separate exclusion artifact and
removed from the main corpus. `excluded_flag` is not carried on retained rows —
by construction every retained row passed the filter.

Deduplication (from `params.yaml`): each token counted once per project, making
TF-IDF reflect document presence rather than within-project repetition.


In [5]:
sq, sp_cfg = CFG["sql"], CFG["preprocess"]

doc_freq = token_doc_freq(df)
keep     = doc_freq[(doc_freq >= sq["min_doc_count"]) &
                    (doc_freq <  len(df) * sq["max_doc_fraction"])].index

dedup = sp_cfg["dedup_tokens_per_project"]
df["tokens"] = df["tokens"].apply(
    lambda ts: list(dict.fromkeys(t for t in ts if t in keep))
    if dedup else [t for t in ts if t in keep]
)

min_tok  = sp_cfg["min_project_tokens"]
excluded = df[df["tokens"].apply(len) < min_tok].copy()
excluded["exclusion_reason"] = f"< {min_tok} tokens after filtering"
df       = df[df["tokens"].apply(len) >= min_tok].reset_index(drop=True)

doc_freq.loc[keep].to_csv(OUT("vocab", "unigram_docfreq.csv"), header=True)
flat_freq(df).to_csv(OUT("vocab", "unigram_freq.csv"), header=True)
excluded.to_parquet(OUT("prepared", "excluded_projects.parquet"), index=False)
df.to_parquet(OUT("prepared", "05_filtered.parquet"), index=False)

print(f"Retained: {len(df):,}  |  Excluded: {len(excluded):,}  |  Vocab: {len(keep):,}")

Retained: 896,277  |  Excluded: 18  |  Vocab: 7,297


---
## Step 6 — Quality Report: Checkpoint 1

SQL refinement loop exit point. Review the quality report and top vocabulary
before proceeding to analysis.


In [6]:
qr1 = quality_report(df, label="cp1", doc_freq=doc_freq.loc[keep],
                     save_path=OUT("quality", "quality_cp1.json"))


=======================================================  [cp1]
  Projects : 896,277
  Tok/proj : min=5  p50=61  max=197
  Vocab    : 7,297 unique tokens
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']



In [7]:
flat_freq(df).head(60).to_frame("count")   # eyeball for junk

,count
work,438552
love,361694
class,323861
project,314409
able,310378
skill,301129
read,299633
allow,299309
create,283919
way,266468
